# IPF transcriptome E: CYL + ZCP Scanpy exploration

Notebook-first workflow adapted from `S12-2N.ipynb`. Confirm QC thresholds, representation, clustering and marker-based annotation here before updating the production Python/Slurm scripts.

In [ ]:
from __future__ import annotations

import json
import tempfile
import zipfile
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 2
sc.set_figure_params(dpi=100, frameon=False)
print('scanpy', sc.__version__)

In [ ]:
PROJECT_DIR = Path('/home/lijia/luozhixiong/IPF_tissue')
INPUT_ZIPS = {
    'CYL': PROJECT_DIR / 'Data/Matrix/25100718_CYL_E/filtered_feature_bc_matrix.zip',
    'ZCP': PROJECT_DIR / 'Data/Matrix/25100718_ZCP_E/filtered_feature_bc_matrix.zip',
}
OUTPUT_DIR = PROJECT_DIR / 'Data/Scanpy/E_CYL_ZCP_notebook'
RANDOM_SEED = 0

for cohort, path in INPUT_ZIPS.items():
    assert path.is_file(), f'Missing input for {cohort}: {path}'
    print(cohort, path, f'{path.stat().st_size / 1024**2:.1f} MiB')

In [ ]:
def read_10x_zip(path: Path, cohort: str) -> ad.AnnData:
    with tempfile.TemporaryDirectory(prefix=f'{cohort}_10x_') as tmp:
        with zipfile.ZipFile(path) as archive:
            archive.extractall(tmp)
        matrix_dir = Path(tmp) / 'filtered_feature_bc_matrix'
        sample = sc.read_10x_mtx(matrix_dir, var_names='gene_symbols', make_unique=True)
    sample.obs_names = pd.Index([f'{cohort}_{barcode}' for barcode in sample.obs_names])
    sample.obs['cohort'] = cohort
    return sample

In [ ]:
samples = {cohort: read_10x_zip(path, cohort) for cohort, path in INPUT_ZIPS.items()}
for cohort, sample in samples.items():
    print(cohort, sample)

adata = ad.concat(list(samples.values()), join='outer', merge='same', index_unique=None)
assert not adata.obs_names.duplicated().any()
adata.obs['cohort'] = adata.obs['cohort'].astype('category')
adata.layers['counts'] = adata.X.copy()
adata

## 1. Explore raw counts and choose QC thresholds
Do not finalize thresholds from the S12 notebook automatically. Compare distributions by cohort and record the chosen cutoffs.

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
adata.obs.groupby('cohort', observed=True)[qc_metrics].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

In [ ]:
sc.pl.violin(adata, qc_metrics, groupby='cohort', jitter=0.25, multi_panel=True)
fig, axs = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', color='cohort', ax=axs[0], show=False)
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color='cohort', ax=axs[1], show=False)

### Candidate thresholds
The values below are starting points from the template, not confirmed project parameters. Adjust them after inspecting the plots and per-cohort pass counts.

In [ ]:
MIN_GENES = 200
MAX_GENES = 6000
MIN_COUNTS = 500
MAX_MT_PERCENT = 5.0
MIN_CELLS_PER_GENE = 3

pass_qc = (
    (adata.obs['n_genes_by_counts'] >= MIN_GENES)
    & (adata.obs['n_genes_by_counts'] < MAX_GENES)
    & (adata.obs['total_counts'] >= MIN_COUNTS)
    & (adata.obs['pct_counts_mt'] < MAX_MT_PERCENT)
)
adata.obs['pass_basic_qc'] = pass_qc
pd.crosstab(adata.obs['cohort'], adata.obs['pass_basic_qc'], margins=True)

In [ ]:
adata_qc = adata[adata.obs['pass_basic_qc']].copy()
sc.pp.filter_genes(adata_qc, min_cells=MIN_CELLS_PER_GENE)
print('before:', adata.shape, 'after:', adata_qc.shape)
adata_qc.obs.groupby('cohort', observed=True)[qc_metrics].median()

## 2. Normalize, select HVGs and inspect PCA
HVGs are selected with `cohort` as the batch key. Cohort correction is not applied automatically because CYL/ZCP may encode biology as well as technical effects.

In [ ]:
N_HVG = 3000
sc.pp.normalize_total(adata_qc, target_sum=1e4)
sc.pp.log1p(adata_qc)
adata_qc.raw = adata_qc
sc.pp.highly_variable_genes(
    adata_qc, n_top_genes=N_HVG, batch_key='cohort', flavor='seurat'
)
sc.pl.highly_variable_genes(adata_qc)
adata_qc.var[['highly_variable', 'highly_variable_nbatches']].value_counts().head()

In [ ]:
REGRESS_COVARIATES = False
adata_work = adata_qc[:, adata_qc.var['highly_variable']].copy()
if REGRESS_COVARIATES:
    sc.pp.regress_out(adata_work, ['total_counts', 'pct_counts_mt'])
sc.pp.scale(adata_work, max_value=10)
sc.tl.pca(adata_work, n_comps=50, svd_solver='arpack', random_state=RANDOM_SEED)
sc.pl.pca_variance_ratio(adata_work, n_pcs=50, log=True)

## 3. Neighbours, UMAP and clustering
Inspect cohort separation before deciding whether a batch-corrected representation is justified. Try parameter alternatives in copied cells rather than overwriting the first result.

In [ ]:
N_PCS = 30
N_NEIGHBORS = 15
sc.pp.neighbors(adata_work, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, random_state=RANDOM_SEED)
sc.tl.umap(adata_work, random_state=RANDOM_SEED)
sc.pl.umap(adata_work, color=['cohort', 'total_counts', 'pct_counts_mt'])

In [ ]:
LEIDEN_RESOLUTION = 0.8
sc.tl.leiden(adata_work, resolution=LEIDEN_RESOLUTION, random_state=RANDOM_SEED)
sc.pl.umap(adata_work, color=['leiden', 'cohort'], legend_loc='on data')
pd.crosstab(adata_work.obs['leiden'], adata_work.obs['cohort'], normalize='index')

## 4. Cluster markers and manual annotation
Do not reuse the S12 cluster-number mapping. Cluster IDs change with filtering, samples and resolution. Review markers and canonical lung markers before filling `cluster_to_cell_type`.

In [ ]:
sc.tl.rank_genes_groups(adata_work, 'leiden', method='wilcoxon', use_raw=True)
sc.pl.rank_genes_groups(adata_work, n_genes=20, sharey=False)
marker_names = pd.DataFrame(adata_work.uns['rank_genes_groups']['names'])
marker_names.head(15)

In [ ]:
marker_genes = {
    'AT1': ['AGER', 'CAV1'],
    'AT2': ['LAMP3', 'SFTPA1', 'SFTPC'],
    'Fibroblast': ['COL1A2', 'DCN', 'COL6A1'],
    'Endothelial': ['PECAM1', 'ENG', 'VWF', 'CDH5'],
    'Monocyte': ['LYZ', 'MS4A7', 'FCN1'],
    'Macrophage': ['CD68', 'MRC1', 'CD163', 'FABP4'],
    'T_NK': ['CD2', 'KLRD1', 'GNLY'],
    'B_cell': ['MS4A1', 'CD79A'],
    'Ciliated': ['FOXJ1', 'DNAH11'],
    'Proliferating': ['MKI67', 'TOP2A'],
}
present_markers = {k: [g for g in genes if g in adata_work.raw.var_names] for k, genes in marker_genes.items()}
sc.pl.dotplot(adata_work, present_markers, groupby='leiden', use_raw=True, dendrogram=True)

In [ ]:
# Fill only after reviewing ranked genes, marker plots and cohort composition.
cluster_to_cell_type = {
    # '0': 'AT2',
}
adata_work.obs['cell_type'] = adata_work.obs['leiden'].map(cluster_to_cell_type).fillna('Unassigned')
sc.pl.umap(adata_work, color=['cell_type', 'cohort'], legend_loc='on data')

## 5. Save only after the analysis choices are confirmed
Set `ANALYSIS_CONFIRMED = True` only after documenting final thresholds, PCA/neighbour settings, cluster resolution and cell-type mapping.

In [ ]:
ANALYSIS_CONFIRMED = False
if not ANALYSIS_CONFIRMED:
    raise RuntimeError('Review and confirm notebook parameters before saving final outputs')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
adata_work.uns['notebook_parameters'] = {
    'min_genes': MIN_GENES, 'max_genes': MAX_GENES, 'min_counts': MIN_COUNTS,
    'max_mt_percent': MAX_MT_PERCENT, 'min_cells_per_gene': MIN_CELLS_PER_GENE,
    'n_hvg': N_HVG, 'regress_covariates': REGRESS_COVARIATES,
    'n_pcs': N_PCS, 'n_neighbors': N_NEIGHBORS,
    'leiden_resolution': LEIDEN_RESOLUTION, 'seed': RANDOM_SEED,
}
adata_work.write_h5ad(OUTPUT_DIR / 'rna_e_cyl_zcp_confirmed.h5ad', compression='gzip')
adata_work.obs[['cohort', 'leiden', 'cell_type']].to_csv(
    OUTPUT_DIR / 'cell_id_cell_type.tsv', sep='\t', index_label='cell_id'
)
with (OUTPUT_DIR / 'confirmed_parameters.json').open('w') as handle:
    json.dump(adata_work.uns['notebook_parameters'], handle, indent=2, sort_keys=True)